# 02 SimMIM Pretraining (v2)

SimMIM-style masked image modeling on a MONAI 2D Swin encoder, adapted to 2D confocal fluorescence microscopy of synaptic puncta.

**Reference paper:** Xie et al., *SimMIM: A Simple Framework for Masked Image Modeling*, CVPR 2022, arXiv:2111.09886.

## Why this notebook exists

| Rung | Default | What it adds |
|------|---------|--------------|
| **A** | yes | image-level split, LR 3e-5, block_size=8, grad_clip=5.0, mask_ratio=0.4, vanilla L1 |
| **B** | flag | foreground-weighted L1 (alpha and tau from train-split stats) |
| **C** | flag | content-aware mask sampling biased to foreground blocks |
| **D** | flag | SwinUNETR-style cutout masking instead of grid masking |

Rung A is the canonical-paper config minus the unavoidable batch-size / data deviations. We move to B/C/D only after A's rank stays healthy.

## Paper-grounded settings

- ℓ1 reconstruction over masked pixels only — Sec 3.4, "An ℓ1-loss is employed on the masked pixels"
- Mask ratio 0.4–0.8 with patch sizes 4/8/16 "smoothly improves accuracy" — Sec 4.1.2 + Table 1
- Predict-only-masked (82.8) > predict-all (82.5) — Table 4
- Light decoder (1×1 Conv + PixelShuffle) matches heavier ones — Table 2
- AdamW, β1=0.9, β2=0.999, weight_decay=0.05 — Sec 4.1.1

## Domain-specific deviations

- LR 3e-5 (paper used 8e-4 with batch 2048; we have batch 32 and very low data diversity, so we apply roughly sqrt-scaling and stay conservative)
- block_size=8 (puncta diameter is 2–5 px; the paper used 32 on 192² ImageNet, but 32 on our 128 sparse data wipes whole neurite neighbourhoods)
- mask_token injected in pixel space rather than after patch_embed (keeps MONAI's SwinTransformer untouched)
- D4 augmentations only; no ImageNet normalisation, no colour jitter
- foreground-aware reconstruction MAE diagnostic alongside standard MAE


## 1. Imports & setup

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import sys
import time
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Locate repo root
REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / "data_utils").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print(f"repo root: {REPO_ROOT}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")

## 2. Configuration

`cfg.rung` selects which rung's defaults are active. Set it to "A", "B", "C", or "D" (or "custom" to override fields manually below).
Setting global seed

In [ ]:
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

cfg = SimpleNamespace(
    seed         = 42,
    output_root  = REPO_ROOT / "outputs",
    tag          = "simmim_v2",
    run_id       = RUN_TS,
    rung         = "A",     # one of A/B/C/D/custom
)

data_cfg = SimpleNamespace(
    patches_dir   = REPO_ROOT / "data" / "patches_128",
    split_json    = REPO_ROOT / "data" / "splits" / "v2_split.json",
    batch_size    = 32,
    num_workers   = 2,
    pin_memory    = True,
)

torch.manual_seed(cfg.seed)

In [ ]:
model_cfg = SimpleNamespace(
    in_channels       = 3,
    spatial_dims      = 2,
    img_size          = 128,
    feature_size      = 48,
    patch_size        = 2,
    window_size       = 7,
    dropout_path_rate = 0.0,
    use_checkpoint    = False,
)

In [ ]:
# Rung defaults
RUNGS = {
    # A. Canonical paper config minus batch-size scaling and pixel-space mask token.
    "A": dict(
        # masking
        mask_ratio        = 0.6,
        mask_block_size   = 8,
        masking_strategy  = "grid",       # 'grid' = SimMIM block grid
        fg_mask_ratio     = 0.0,          # 0.0 = pure uniform sampling
        # loss
        loss_kind         = "l1",         # 'l1' | 'l1_fg' | 'l1_l2_mix'
        fg_alpha_override = None,         # None = use split JSON
        # optimizer
        lr                = 3e-5,
        weight_decay      = 0.05,
        warmup_epochs     = 10,
        epochs            = 200,
        grad_clip_norm    = 5.0,
    ),
    # B. + foreground-weighted L1.
    "B": dict(
        mask_ratio        = 0.1,
        mask_block_size   = 8,
        masking_strategy  = "grid",
        fg_mask_ratio     = 0.0,
        loss_kind         = "l1_fg",
        fg_alpha_override = None,
        lr                = 3e-5,
        weight_decay      = 0.05,
        warmup_epochs     = 10,
        epochs            = 200,
        grad_clip_norm    = 5.0,
    ),
    # C. + content-aware masking. 50% of masked area on foreground blocks.
    "C": dict(
        mask_ratio        = 0.25,
        mask_block_size   = 8,
        masking_strategy  = "content_aware",
        fg_mask_ratio     = 0.5,
        loss_kind         = "l1_fg",
        fg_alpha_override = None,
        lr                = 3e-5,
        weight_decay      = 0.05,
        warmup_epochs     = 10,
        epochs            = 200,
        grad_clip_norm    = 5.0,
    ),
    # D. + SwinUNETR-style cutout masking. Tang et al. 2022, Sec 5.2: 30% drop rate
    # via random rectangles. Uses ratio = 0.3 to match the paper.
    "D": dict(
        mask_ratio        = 0.3,
        mask_block_size   = 8,            # used as min cutout side
        masking_strategy  = "cutout",
        fg_mask_ratio     = 0.0,
        loss_kind         = "l1_fg",
        fg_alpha_override = None,
        lr                = 3e-5,
        weight_decay      = 0.05,
        warmup_epochs     = 10,
        epochs            = 200,
        grad_clip_norm    = 5.0,
    ),
}

train_cfg = SimpleNamespace(**RUNGS[cfg.rung])
print(f"rung {cfg.rung}:")
for k, v in vars(train_cfg).items():
    print(f"  {k:20s} {v}")


## 3. Dataset (image-level split)

Loads the v2 split produced by `00_data_and_split.ipynb`. The fields `fg_stats.tau` and `fg_stats.alpha` are pre-computed once on the train split and re-used here.


In [ ]:
from data_utils.split_patch_dataset import SplitPatchDataset

assert data_cfg.split_json.exists(), \
    f"{data_cfg.split_json} not found. Run 00_data_and_split.ipynb first."

ds_train = SplitPatchDataset(data_cfg.patches_dir, data_cfg.split_json, which="train")
ds_val   = SplitPatchDataset(data_cfg.patches_dir, data_cfg.split_json, which="val")

train_loader = DataLoader(
    ds_train, batch_size=data_cfg.batch_size, shuffle=True,
    num_workers=data_cfg.num_workers, persistent_workers=(data_cfg.num_workers > 0), pin_memory=data_cfg.pin_memory,
    drop_last=True,
)
val_loader = DataLoader(
    ds_val, batch_size=data_cfg.batch_size, shuffle=False,
    num_workers=data_cfg.num_workers, persistent_workers=(data_cfg.num_workers > 0), pin_memory=data_cfg.pin_memory,
    drop_last=False,
)
print(f"train: {len(ds_train):,} patches | val: {len(ds_val):,} patches")
print(f"fg_stats: tau={ds_train.fg_stats['tau']:.5f}  alpha={ds_train.fg_stats['alpha']:.2f}")

# Foreground threshold and weighting scalar.
TAU   = float(ds_train.fg_stats["tau"])
ALPHA = float(train_cfg.fg_alpha_override) if train_cfg.fg_alpha_override is not None \
        else float(ds_train.fg_stats["alpha"])
print(f"using TAU={TAU:.5f}  ALPHA={ALPHA:.2f}")


## 4. Model: `SimMIMSwin`

MONAI 2D `SwinTransformer` backbone + 1×1 Conv + PixelShuffle decoder (SimMIM "linear" head). The mask token is injected in pixel space (DEVIATION) so the upstream MONAI module is unmodified.

The deepest stage of `SwinTransformer` on a 128×128 input has shape `(B, 768, 4, 4)` for our `feature_size=48` (verified empirically). `encoder_stride = patch_size · 2^4 = 32`.


In [ ]:
from monai.networks.nets.swin_unetr import SwinTransformer


class SimMIMSwin(nn.Module):
    """SimMIM head over a MONAI 2D Swin encoder."""

    def __init__(self, args):
        super().__init__()
        self.in_chans       = args.in_channels
        self.img_size       = args.img_size
        self.encoder_stride = args.patch_size * (2 ** 4)
        assert args.img_size % self.encoder_stride == 0

        patch_size  = (args.patch_size,)  * args.spatial_dims
        window_size = (args.window_size,) * args.spatial_dims

        self.swinViT = SwinTransformer(
            in_chans       = args.in_channels,
            embed_dim      = args.feature_size,
            window_size    = window_size,
            patch_size     = patch_size,
            depths         = [2, 2, 2, 2],
            num_heads      = [3, 6, 12, 24],
            mlp_ratio      = 4.0,
            qkv_bias       = True,
            drop_rate      = 0.0,
            attn_drop_rate = 0.0,
            drop_path_rate = args.dropout_path_rate,
            norm_layer     = nn.LayerNorm,
            use_checkpoint = args.use_checkpoint,
            spatial_dims   = args.spatial_dims,
        )

        enc_ch = args.feature_size * (2 ** 4)

        # SimMIM "linear" head; ablation Table 2 shows heavier decoders match.
        self.decoder = nn.Sequential(
            nn.Conv2d(enc_ch, self.encoder_stride ** 2 * self.in_chans, kernel_size=1),
            nn.PixelShuffle(self.encoder_stride),
        )

        # DEVIATION: pixel-space mask token. Canonical SimMIM uses an embed_dim
        # vector after patch_embed; we use a per-channel scalar broadcast over
        # masked pixels to avoid editing MONAI's SwinTransformer.
        self.mask_token = nn.Parameter(torch.zeros(1, self.in_chans, 1, 1))
        nn.init.trunc_normal_(self.mask_token, mean=0.0, std=0.02)

    def encode(self, x):
        return self.swinViT(x.contiguous())[4]

    def encode_pooled(self, x):
        return self.encode(x).mean(dim=(2, 3))

    def forward(self, x, mask):
        x_masked = x * (1.0 - mask) + self.mask_token * mask
        z = self.encode(x_masked)
        return self.decoder(z)

In [ ]:
model = SimMIMSwin(model_cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"SimMIMSwin: {n_params/1e6:.2f}M params")

with torch.no_grad():
    _x = torch.zeros(1, model_cfg.in_channels, model_cfg.img_size, model_cfg.img_size).to(device)
    _z = model.encode(_x)
    print(f"bottleneck shape: {tuple(_z.shape)}  (expected (1, 768, 4, 4))")

## 5. Reconstruction losses

Three options, selected by `train_cfg.loss_kind`:

| `loss_kind` | Formula | Source |
|---|---|---|
| `l1` | mean(|pred - target| · mask) over masked pixel-channels | SimMIM Sec 3.4 |
| `l1_fg` | mean((1 + alpha · 1[target > tau]) · |pred - target| · mask) | DEVIATION (fluorescence). Counters predict-zero attractor. See `docs/fluorescence_ssl_design.md` Deviation 1. |
| `l1_l2_mix` | 0.5 · L1 + 0.5 · L2 over masked pixels | UniFMIR Sec "Training losses" (Ma et al. 2024) |

Loss is computed over masked positions only because SimMIM Table 4 shows predict-only-masked > predict-all (82.8 vs 82.5).


In [ ]:
def simmim_l1_loss(pred, target, mask):
    """Vanilla L1 over masked pixels (SimMIM Sec 3.4)."""
    err   = (pred - target).abs() * mask                     # (B,C,H,W)
    denom = mask.sum() * pred.shape[1] + 1e-8
    return err.sum() / denom


def simmim_l1_fg_loss(pred, target, mask, tau: float, alpha: float):
    """Foreground-weighted L1.

    weight(x) = 1 + alpha * 1[target > tau], applied per pixel-channel.
    With alpha tuned so foreground and background contribute equal total
    weight, this removes the predict-zero attractor on sparse images.

    DEVIATION (fluorescence). See docs/fluorescence_ssl_design.md Deviation 1.
    """
    fg_mask = (target > tau).float()                         # (B,C,H,W)
    weight  = 1.0 + alpha * fg_mask                          # (B,C,H,W)
    err     = (pred - target).abs() * mask * weight          # (B,C,H,W)
    denom   = (mask * weight).sum() + 1e-8                   # mask is (B,1,H,W); broadcast
    # mask is (B,1,H,W); broadcast multiply with weight (B,C,H,W) yields (B,C,H,W).
    # Use mask sum across channels via the same broadcast in denom.
    return err.sum() / denom


def simmim_l1_l2_mix_loss(pred, target, mask):
    """L = 0.5 * L1 + 0.5 * L2 over masked pixels (UniFMIR Sec 'Training losses')."""
    diff  = (pred - target) * mask
    l1    = diff.abs().sum()
    l2    = (diff * diff).sum()
    denom = mask.sum() * pred.shape[1] + 1e-8
    return 0.5 * (l1 / denom) + 0.5 * (l2 / denom)


def reconstruction_loss(pred, target, mask, kind: str, tau: float, alpha: float):
    if kind == "l1":
        return simmim_l1_loss(pred, target, mask)
    if kind == "l1_fg":
        return simmim_l1_fg_loss(pred, target, mask, tau, alpha)
    if kind == "l1_l2_mix":
        return simmim_l1_l2_mix_loss(pred, target, mask)
    raise ValueError(f"unknown loss_kind: {kind!r}")


## 6. Mask sampling

- `grid`: SimMIM block grid masking (default).
- `content_aware`: SimMIM grid, but at least `fg_mask_ratio` of masked blocks contain foreground (mean(target>tau) >= 0.05). Falls back to uniform if no foreground blocks exist.
- `cutout`: SwinUNETR-style random rectangular cutouts summing to ~mask_ratio of the image area. Tang et al. 2022, Sec 5.2.

All return `(B, 1, H, W)` float tensors with 1 = masked, broadcast over channels.


In [ ]:
def random_block_mask(img, block_size: int, mask_ratio: float):
    """SimMIM block grid mask. Vectorised across the batch."""
    B, _, H, W = img.shape
    assert H % block_size == 0 and W % block_size == 0, \
        f"H={H},W={W} not divisible by block_size={block_size}"
    gh, gw   = H // block_size, W // block_size
    n_blocks = gh * gw
    n_mask   = int(math.ceil(n_blocks * mask_ratio))
    noise    = torch.rand(B, n_blocks, device=img.device)
    rank     = noise.argsort(dim=1)
    flat     = (rank < n_mask).float()
    m        = flat.view(B, 1, gh, gw)
    return F.interpolate(m, scale_factor=block_size, mode="nearest")


def content_aware_block_mask(
    img, block_size: int, mask_ratio: float,
    tau: float, fg_mask_ratio: float = 0.5, fg_block_thresh: float = 0.05,
):
    """SimMIM grid masking biased to foreground.

    Sample `n_mask = ceil(n_blocks * mask_ratio)` blocks, but ensure at least
    `floor(n_mask * fg_mask_ratio)` of them are foreground blocks (a block is
    foreground if mean(any-channel > tau) >= fg_block_thresh). If an image has
    fewer foreground blocks than the quota, fill the rest from any-block uniform.

    DEVIATION (fluorescence). See docs/fluorescence_ssl_design.md Deviation 2.
    """
    B, C, H, W = img.shape
    assert H % block_size == 0 and W % block_size == 0
    gh, gw   = H // block_size, W // block_size
    n_blocks = gh * gw
    n_mask   = int(math.ceil(n_blocks * mask_ratio))
    n_fg_q   = int(math.floor(n_mask * fg_mask_ratio))

    # Per-block foreground indicator: any pixel of any channel above tau within the block.
    fg_pix      = (img > tau).any(dim=1, keepdim=True).float()          # (B,1,H,W)
    fg_block    = F.avg_pool2d(fg_pix, kernel_size=block_size)          # (B,1,gh,gw)
    fg_block    = (fg_block >= fg_block_thresh).view(B, n_blocks)       # (B, n_blocks)

    flat = torch.zeros(B, n_blocks, device=img.device)
    for b in range(B):
        fg_idx = fg_block[b].nonzero(as_tuple=False).flatten()
        bg_idx = (~fg_block[b]).nonzero(as_tuple=False).flatten()
        n_fg_take = min(n_fg_q, fg_idx.numel())
        n_bg_take = n_mask - n_fg_take
        # Foreground picks
        if n_fg_take > 0:
            perm = fg_idx[torch.randperm(fg_idx.numel(), device=img.device)[:n_fg_take]]
            flat[b, perm] = 1
        # Background picks
        if n_bg_take > 0 and bg_idx.numel() > 0:
            perm = bg_idx[torch.randperm(bg_idx.numel(), device=img.device)[:n_bg_take]]
            flat[b, perm] = 1
        # If we still owe blocks (background pool too small), fill from anywhere.
        deficit = n_mask - int(flat[b].sum().item())
        if deficit > 0:
            avail = (flat[b] == 0).nonzero(as_tuple=False).flatten()
            perm = avail[torch.randperm(avail.numel(), device=img.device)[:deficit]]
            flat[b, perm] = 1

    m = flat.view(B, 1, gh, gw)
    return F.interpolate(m, scale_factor=block_size, mode="nearest")


def cutout_mask(img, min_side: int, mask_ratio: float, max_tries: int = 64):
    """Random rectangular cutouts summing to ~mask_ratio of the image area.

    Tang et al. 2022 (SwinUNETR SSL), Sec 5.2: drop rate 30% via randomly
    generated regions. We implement with rectangles of side >= min_side
    sampled uniformly until total masked area >= target.
    """
    B, _, H, W = img.shape
    target = int(round(mask_ratio * H * W))
    out = torch.zeros(B, 1, H, W, device=img.device)
    for b in range(B):
        masked = 0
        tries = 0
        while masked < target and tries < max_tries:
            sh = int(torch.randint(min_side, max(min_side + 1, H // 2), (1,)).item())
            sw = int(torch.randint(min_side, max(min_side + 1, W // 2), (1,)).item())
            y0 = int(torch.randint(0, H - sh + 1, (1,)).item())
            x0 = int(torch.randint(0, W - sw + 1, (1,)).item())
            patch = out[b, 0, y0:y0+sh, x0:x0+sw]
            new_pixels = (patch == 0).sum().item()
            patch.fill_(1.0)
            masked += new_pixels
            tries  += 1
    return out


def make_mask(img, train_cfg_, tau: float):
    s = train_cfg_.masking_strategy
    if s == "grid":
        return random_block_mask(img, train_cfg_.mask_block_size, train_cfg_.mask_ratio)
    if s == "content_aware":
        return content_aware_block_mask(
            img, train_cfg_.mask_block_size, train_cfg_.mask_ratio,
            tau=tau, fg_mask_ratio=train_cfg_.fg_mask_ratio,
        )
    if s == "cutout":
        return cutout_mask(img, train_cfg_.mask_block_size, train_cfg_.mask_ratio)
    raise ValueError(f"unknown masking_strategy: {s!r}")


## 7. D4 augmentations

Microscopy has no privileged orientation, so all 8 elements of the dihedral group D4 (4 rotations × 2 flips) are valid augmentations. Applied before masking.


In [ ]:
def augment_batch(x):
    """In-place per-sample D4 augmentations."""
    B = x.shape[0]
    do_h  = torch.rand(B) > 0.5
    do_v  = torch.rand(B) > 0.5
    rot_k = torch.randint(0, 4, (B,))
    for i in range(B):
        if do_h[i]:
            x[i] = x[i].flip(-1)
        if do_v[i]:
            x[i] = x[i].flip(-2)
        k = int(rot_k[i].item())
        if k > 0:
            x[i] = torch.rot90(x[i], k, dims=(-2, -1))
    return x


## 8. Sanity check: masked sample

In [ ]:
import matplotlib.pyplot as plt

_x = next(iter(train_loader)).to(device)
_x = augment_batch(_x.clone())
_m = make_mask(_x, train_cfg, TAU)
print(f"mask coverage: {_m.mean().item():.4f}  (expected ~{train_cfg.mask_ratio})")

In [ ]:

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
img0 = _x[0].amax(0).cpu().numpy()
m0   = _m[0, 0].cpu().numpy()
axes[0].imshow(img0, cmap="gray", vmin=0, vmax=1); axes[0].set_title("input (max-proj)")
axes[1].imshow(m0,  cmap="gray", vmin=0, vmax=1); axes[1].set_title(f"mask ({train_cfg.masking_strategy})")
axes[2].imshow(img0 * (1 - m0), cmap="gray", vmin=0, vmax=1); axes[2].set_title("masked")
axes[3].imshow((img0 > TAU).astype(float), cmap="gray", vmin=0, vmax=1); axes[3].set_title(f"fg (>tau={TAU:.3f})")
for a in axes: a.axis("off")
fig.tight_layout(); plt.show()

## V3 Sanity checks (pre-training)

Uses the **same fixed patch indices** so reconstructions are directly comparable
across notebooks. Verifies dataset shape/dtype/normalization, the actual
augmented inputs the model will see, and (for masked methods) that the mask
matches the configured ratio.


In [ ]:
# === V3-SANITY-CHECKS PRE ===
# Pre-training sanity checks. Indices match the reference notebook
# notebooks/training/pretrain_simMIM_swin_v2_fixed.ipynb so plots are
# directly comparable across runs.
import matplotlib.pyplot as plt
import numpy as np
import torch

from types import SimpleNamespace
overfit_cfg    = SimpleNamespace(patch_indices=[3, 4, 50, 1000])
unseen_idx     = 100
unseen_indices = [200, 500, 1500, 2000]

def _resolve_fixed(ds, indices):
    """Return a (len(indices), C, H, W) tensor of the requested patches.

    SplitPatchDataset stores the underlying PatchDataset as `_base`, so we
    bypass the split filter to get the same absolute patches the reference
    notebook uses (matches `dataset[i]` in pretrain_simMIM_swin_v2_fixed.ipynb).
    """
    base = getattr(ds, "_base", ds)
    return torch.stack([base[i] for i in indices]).to(device)


x_fixed = _resolve_fixed(ds_train, overfit_cfg.patch_indices)
print(f"x_fixed: shape={tuple(x_fixed.shape)}  dtype={x_fixed.dtype}  "
      f"device={x_fixed.device}")

# Per-channel intensity stats on a real train batch -- catches channel
# misalignment (e.g. a channel that is identically zero) and abnormal
# normalisation. Reference: cells 4 and 21 of pretrain_simMIM_swin_v2_fixed.ipynb.
_x = next(iter(train_loader)).to(device)
print(f"\ntrain batch:  shape={tuple(_x.shape)}  dtype={_x.dtype}  "
      f"min={_x.min():.3f}  max={_x.max():.3f}")
for c in range(_x.shape[1]):
    xc = _x[:, c]
    print(f"  ch{c}:  min={xc.min():.4f}  max={xc.max():.4f}  "
          f"mean={xc.mean():.4f}  std={xc.std():.4f}")

assert _x.ndim == 4, f"expected (B,C,H,W), got {_x.shape}"
assert _x.shape[1] == model_cfg.in_channels, (
    f"channel mismatch: dataset={_x.shape[1]} model={model_cfg.in_channels}"
)
assert _x.shape[2] == model_cfg.img_size and _x.shape[3] == model_cfg.img_size, (
    f"spatial mismatch: dataset={_x.shape[2:]} model={model_cfg.img_size}"
)
assert -1e-6 <= float(_x.min()) and float(_x.max()) <= 1.0 + 1e-6, "values outside [0,1]"

# Foreground sparsity diagnostic. With tau ~ 0.66 and alpha = 99 the loss
# is dominated by the tiny fraction of foreground pixels; this prints the
# realised fraction so the bug is visible at glance.
try:
    _fg = (_x.amax(dim=1, keepdim=True) >= TAU).float()
    print(f"  foreground fraction (>= TAU={float(TAU):.4f}): "
          f"{_fg.mean().item()*100:.2f}%")
except (NameError, RuntimeError):
    pass

# --- Visualise the 4 fixed patches: per-channel slices + RGB composite
n_idx = x_fixed.shape[0]; C = x_fixed.shape[1]
fig, axes = plt.subplots(C + 1, n_idx, figsize=(3.0 * n_idx, 3.0 * (C + 1)))
if axes.ndim == 1:
    axes = axes[None, :]
for j in range(n_idx):
    img = x_fixed[j].cpu()
    for c in range(C):
        axes[c, j].imshow(img[c].numpy(), cmap="gray", vmin=0, vmax=1)
        axes[c, j].set_title(f"idx={overfit_cfg.patch_indices[j]}  ch{c}")
        axes[c, j].axis("off")
    if C >= 3:
        rgb = img[:3].clamp(0, 1).permute(1, 2, 0).numpy()
    else:
        rgb = np.repeat(img.numpy(), 3, axis=0).transpose(1, 2, 0).clip(0, 1)
    axes[C, j].imshow(rgb)
    axes[C, j].set_title("RGB" if C >= 3 else "expanded gray")
    axes[C, j].axis("off")
plt.suptitle(f"Pre-train fixed indices: {overfit_cfg.patch_indices}")
plt.tight_layout(); plt.show()

# --- Mask + masked input on the same fixed indices
torch.manual_seed(cfg.seed)
x_aug = augment_batch(x_fixed.clone())
mask_fixed = make_mask(x_aug, train_cfg, TAU)
masked_input = x_aug * (1.0 - mask_fixed)
print(f"effective mask coverage: {mask_fixed.mean().item():.4f}  "
      f"(configured mask_ratio={train_cfg.mask_ratio})")
assert abs(mask_fixed.mean().item() - train_cfg.mask_ratio) < 0.20, (
    f"mask coverage drifted: realised={mask_fixed.mean().item():.3f} "
    f"target={train_cfg.mask_ratio:.3f}")

n_idx = x_fixed.shape[0]
fig, axes = plt.subplots(3, n_idx, figsize=(3.0 * n_idx, 9.0))
for j in range(n_idx):
    a = x_aug[j].clamp(0, 1).cpu()
    m = mask_fixed[j, 0].cpu()
    mi = masked_input[j].clamp(0, 1).cpu()
    axes[0, j].imshow(a.permute(1, 2, 0).numpy() if a.shape[0] >= 3 else a[0].numpy(), cmap="gray")
    axes[0, j].set_title(f"aug idx={overfit_cfg.patch_indices[j]}"); axes[0, j].axis("off")
    axes[1, j].imshow(m.numpy(), cmap="gray", vmin=0, vmax=1)
    axes[1, j].set_title(f"mask cov={mask_fixed[j].mean().item():.2f}"); axes[1, j].axis("off")
    axes[2, j].imshow(mi.permute(1, 2, 0).numpy() if mi.shape[0] >= 3 else mi[0].numpy(), cmap="gray")
    axes[2, j].set_title("masked input"); axes[2, j].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# --- End-to-end forward shape check (no backprop)
model.eval()
with torch.no_grad():
    out = model(x_aug, mask_fixed)
    _r = out[0] if isinstance(out, tuple) else out
    _z = None
print(f"recon shape: {tuple(_r.shape)}  dtype={_r.dtype}")
if _z is not None:
    print(f"z_pool shape: {tuple(_z.shape)}  dtype={_z.dtype}")
assert _r.shape == x_fixed.shape, f"recon {_r.shape} != input {x_fixed.shape}"

In [ ]:
# === V4-RECON-VIZ-HELPERS ===
# Single visualisation function used after the overfit-on-batch sanity
# check (cell below) AND after full training (section 14), on the SAME
# 4 overfit patches + 3 unseen random train patches both times. The two
# figures are directly comparable -- if the final-training figure is no
# better than the overfit one, the full run did not learn anything that
# 4-patch overfit didn't already memorise.
#
# This cell also picks the 3 unseen indices + their masks once, so all
# downstream cells share the same inputs (deterministic via cfg.seed).
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch


def _resolve_patches(ds, indices, device):
    """Bypass the split filter on SplitPatchDataset and return a
    (len(indices), C, H, W) tensor of the requested absolute patches."""
    base = getattr(ds, "_base", ds)
    return torch.stack([base[int(i)] for i in indices]).to(device)


# Deterministic pick of 3 unseen random patches (excludes the overfit
# indices). Reused by both the overfit-recon-viz cell and the post-
# training viz cell so the two figures show the same patches.
_excluded = set(int(i) for i in overfit_cfg.patch_indices)
_base_train = getattr(ds_train, "_base", ds_train)
_rng = np.random.default_rng(cfg.seed + 17)
_pool = [i for i in range(len(_base_train)) if i not in _excluded]
viz_other_indices = [int(x) for x in _rng.choice(_pool, size=3, replace=False)]
x_other = _resolve_patches(ds_train, viz_other_indices, device)

# Deterministic mask for x_other so the SAME pixels are masked in the
# overfit-viz and the final-viz -- crucial for fair comparison.
torch.manual_seed(cfg.seed + 23)
mask_other = make_mask(x_other, train_cfg, TAU)
print(f"viz: fixed indices = {overfit_cfg.patch_indices}  "
      f"unseen indices = {viz_other_indices}  "
      f"(out of {len(_base_train)} train patches)")


def viz_recon(model, x, mask, indices, suptitle, *, header=None,
              autocast=True):
    """Run inference, print per-patch in/out-of-mask MAE, then plot a
    4-row grid: input | mask | recon | |err| with a cyan mask contour
    on the |err| row and a per-axis colorbar (vmax shared across the
    batch so colours mean the same thing across patches).

    Pure -- does not modify the model state beyond the eval/train flag.
    Caller is responsible for any state save/restore (see overfit cell).
    """
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad(), torch.amp.autocast(
            device.type, enabled=autocast and device.type == "cuda"
        ):
            out = model(x, mask)
            recon = (out[0] if isinstance(out, tuple) else out).float()
    finally:
        if was_training:
            model.train()

    err = (recon - x).abs()
    if header:
        print(header)
    for j, idx in enumerate(indices):
        in_m = mask[j, 0]
        e    = err[j].mean(0)
        in_mae  = (e * in_m).sum() / max(in_m.sum().item(), 1.0)
        out_mae = (e * (1.0 - in_m)).sum() / max((1.0 - in_m).sum().item(), 1.0)
        print(f"  idx={idx}: in-mask MAE={in_mae.item():.5f}  "
              f"out-of-mask MAE={out_mae.item():.5f}  "
              f"full MAE={err[j].mean().item():.5f}")

    err_max = max(float(err.max().item()), 1e-6)
    n = x.shape[0]
    fig, axes = plt.subplots(4, n, figsize=(3.6 * n, 12.0))
    if n == 1:
        axes = axes[:, None]
    for j in range(n):
        o  = x[j].clamp(0, 1).cpu()
        rr = recon[j].clamp(0, 1).cpu()
        mm = mask[j, 0].cpu()
        ee = err[j].mean(0).cpu()
        o_disp = o.permute(1, 2, 0).numpy()  if o.shape[0]  >= 3 else o[0].numpy()
        r_disp = rr.permute(1, 2, 0).numpy() if rr.shape[0] >= 3 else rr[0].numpy()

        axes[0, j].imshow(o_disp, cmap="gray")
        axes[0, j].set_title(f"input idx={indices[j]}")
        axes[0, j].axis("off")

        axes[1, j].imshow(mm.numpy(), cmap="gray", vmin=0, vmax=1)
        axes[1, j].set_title(f"mask cov={mask[j].mean().item():.2f}")
        axes[1, j].axis("off")

        axes[2, j].imshow(r_disp, cmap="gray")
        axes[2, j].set_title("recon")
        axes[2, j].axis("off")

        im = axes[3, j].imshow(ee.numpy(), cmap="hot",
                               vmin=0.0, vmax=err_max)
        axes[3, j].contour(mm.numpy(), levels=[0.5],
                           colors=["cyan"], linewidths=1.0)
        in_mae  = (ee * mm).sum() / max(mm.sum().item(), 1.0)
        out_mae = (ee * (1.0 - mm)).sum() / max((1.0 - mm).sum().item(), 1.0)
        axes[3, j].set_title(
            f"|err|  in={in_mae.item():.3f}  out={out_mae.item():.3f}"
        )
        axes[3, j].axis("off")
        cb = plt.colorbar(im, ax=axes[3, j], fraction=0.046, pad=0.04)
        cb.set_label("|err|", fontsize=8)
        cb.ax.tick_params(labelsize=7)
    plt.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

In [ ]:
# === V4-OVERFIT-ON-BATCH ===
# Overfit-on-a-batch sanity check (matches reference cells 25-29 of
# pretrain_simMIM_swin_v2_fixed.ipynb).
#
# This step trains the *current* model on the 4 fixed patches for a short
# burst with a saved/restored state_dict so the main training run starts
# from the same weights it would have without this cell. If the loss does
# not collapse on 4 patches in ~200 steps the model+loss is broken
# regardless of how the full training looks.
import copy

RUN_OVERFIT_CHECK   = True   # set to False to skip
N_OVERFIT_STEPS     = 2000
OVERFIT_LR          = 1e-4

In [ ]:
if RUN_OVERFIT_CHECK:
    _saved = copy.deepcopy(model.state_dict())
    opt = torch.optim.AdamW(model.parameters(), lr=OVERFIT_LR, weight_decay=0.0)
    losses = []
    model.train()
    for step in range(N_OVERFIT_STEPS):
        out = model(x_fixed, mask_fixed)
        r   = out[0] if isinstance(out, tuple) else out
        diff = (r - x_fixed).abs()
        denom = mask_fixed.sum() * x_fixed.shape[1] + 1e-8
        loss = (diff * mask_fixed).sum() / denom
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        losses.append(float(loss.item()))

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(losses)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_yscale("log")
    ax.set_title(
        f"overfit-on-{x_fixed.shape[0]}-patches  "
        f"start={losses[0]:.4f}  end={losses[-1]:.4f}"
    )
    plt.tight_layout(); plt.show()

    drop = (losses[0] - losses[-1]) / max(losses[0], 1e-8)
    print(f"loss drop over {N_OVERFIT_STEPS} steps: {drop*100:.1f}%  ")
    if drop < 0.50:
        print("  WARNING: model failed to overfit a tiny batch; investigate "
              "loss / mask / fp16 overflow before launching full training.")
else:
    print("overfit-on-batch sanity check skipped (RUN_OVERFIT_CHECK=False)")

In [ ]:
# === V4-OVERFIT-RECON-VIZ ===
# Visualise what the overfit-on-batch model learned: reconstructions on
# the same 4 patches it was trained on (expect near-perfect on masked
# pixels) plus 3 unseen random train-split patches (expect poor recon --
# a 4-patch overfit is OOD on the rest of the dataset). The same
# `viz_recon()` helper is reused in section 14 after full training, on
# the SAME indices, so the two figures are directly comparable.
#
# Implementation: replay the overfit loop with hyperparameters from the
# previous cell, run the viz on both groups, then restore the model
# state. The replay is short (~10 s) and self-contained so the main
# training run starts from the same weights either way.
if RUN_OVERFIT_CHECK:
    _saved_pre = copy.deepcopy(model.state_dict())
    opt = torch.optim.AdamW(model.parameters(), lr=OVERFIT_LR, weight_decay=0.0)
    model.train()
    for step in range(N_OVERFIT_STEPS):
        out = model(x_fixed, mask_fixed)
        r   = out[0] if isinstance(out, tuple) else out
        diff = (r - x_fixed).abs()
        denom = mask_fixed.sum() * x_fixed.shape[1] + 1e-8
        loss = (diff * mask_fixed).sum() / denom
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
    print(f"replayed overfit final loss: {float(loss.item()):.5f}  "
          f"(over {N_OVERFIT_STEPS} steps at lr={OVERFIT_LR})")

    viz_recon(
        model, x_fixed, mask_fixed, overfit_cfg.patch_indices,
        suptitle=f"OVERFIT recon -- {x_fixed.shape[0]} TRAIN patches "
                 f"(expect near-perfect on masked pixels)",
        header="\nOVERFIT patches (expect near-perfect on masked pixels):",
    )

    viz_recon(
        model, x_other, mask_other, viz_other_indices,
        suptitle="OVERFIT recon -- 3 UNSEEN random train patches "
                 "(expect poor recon: 4-patch overfit is OOD)",
        header="\nUNSEEN random patches (expect poor recon -- 4-patch overfit, OOD):",
    )

    model.load_state_dict(_saved_pre)
    print("\nmodel state restored to pre-overfit weights")
else:
    print("overfit-recon visualisation skipped (RUN_OVERFIT_CHECK=False)")

In [ ]:
if RUN_OVERFIT_CHECK:
    model.load_state_dict(_saved)
    print("model state restored to its pre-sanity weights")

## 10. Diagnostics

We log five numbers per check:

- **eff_rank** of channel-pooled features over ~256 val samples (entropy of normalised squared singular values). Goes to D when the encoder uses every dimension. Drops to ~1 when the encoder collapses.
- **feat_std** min/median/max across the 768 dimensions.
- **n_dead** dimensions with std < 1e-3.
- **recon_mae_all**: mean absolute error on masked pixels (any intensity).
- **recon_mae_fg**: mean absolute error on masked pixels with `target > tau`. This is the metric that detects predict-zero (it stays high when the encoder predicts ~0 even though `recon_mae_all` looks fine).


In [ ]:
import itertools

N_DIAG_BATCHES = 8

In [ ]:
def collect_diag_features(model, loader, n=N_DIAG_BATCHES):
     feats = []
     with torch.no_grad():
         it = iter(loader)
         try:
             for _ in range(n):
                 try:
                     x = next(it)
                 except StopIteration:
                     break
                 x = x.to(device, non_blocking=True)
                 feats.append(model.encode_pooled(x).cpu())
         finally:
             del it
     return torch.cat(feats, dim=0)



def collapse_metrics(feats):
    D = feats.shape[1]
    r_max = min(feats.shape[0] - 1, D)
    feat_std = feats.std(dim=0)
    n_dead = (feat_std < 1e-3).sum().item()
    fc = feats - feats.mean(dim=0, keepdim=True)
    s = torch.linalg.svdvals(fc.float())
    s2 = (s ** 2) / ((s ** 2).sum() + 1e-12)
    eff_rank = torch.exp(-(s2 * torch.log(s2 + 1e-12)).sum()).item()
    return dict(
        eff_rank=eff_rank, r_max=r_max, n_dead=n_dead, D=D,
        std_min=float(feat_std.min()), std_med=float(feat_std.median()),
        std_max=float(feat_std.max()),
    )


def recon_mae_split(model, loader, train_cfg_, tau: float, n_batches: int = 4):
    """Compute MAE over masked pixels, split by foreground/background."""
    model.eval()
    s_all, n_all = 0.0, 0
    s_fg,  n_fg  = 0.0, 0
    with torch.no_grad():
        for x in itertools.islice(loader, n_batches):
            x = x.to(device, non_blocking=True)
            mask = make_mask(x, train_cfg_, tau)
            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                recon = model(x, mask)
            err  = (recon.float() - x.float()).abs()
            mexp = mask.expand_as(err).bool()
            s_all += err[mexp].sum().item();   n_all += mexp.sum().item()
            fg = (x > tau) & mexp
            s_fg  += err[fg].sum().item();     n_fg  += fg.sum().item()
    mae_all = s_all / max(1, n_all)
    mae_fg  = s_fg / max(1, n_fg)
    return dict(mae_all=mae_all, mae_fg=mae_fg, n_fg=n_fg)

In [ ]:
# Pre-training baseline
diag0 = collapse_metrics(collect_diag_features(model, val_loader))
mae0  = recon_mae_split(model, val_loader, train_cfg, TAU)
print(
    f"epoch 0 | eff_rank {diag0['eff_rank']:.1f}/{diag0['r_max']} "
    f"| std med {diag0['std_med']:.3f} | dead {diag0['n_dead']}/{diag0['D']} "
    f"| mae_all {mae0['mae_all']:.5f} | mae_fg {mae0['mae_fg']:.5f}"
)


## 11. Output directory + logging

In [ ]:
save_dir = cfg.output_root / f"{cfg.tag}_{cfg.rung}_{cfg.run_id}"
save_dir.mkdir(parents=True, exist_ok=True)
print(f"save_dir: {save_dir}")

with (save_dir / "config.json").open("w") as f:
    json.dump(dict(
        cfg       = {k: str(v) for k, v in vars(cfg).items()},
        data_cfg  = {k: str(v) for k, v in vars(data_cfg).items()},
        model_cfg = vars(model_cfg),
        train_cfg = vars(train_cfg),
        TAU=TAU, ALPHA=ALPHA,
    ), f, indent=2)

csv_path = save_dir / "history.csv"
csv_file = csv_path.open("w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=[
    "epoch", "train_loss", "val_loss", "lr", "grad_norm_mean", "grad_norm_max",
    "epoch_time_s", "eff_rank", "r_max", "n_dead", "std_med",
    "mae_all", "mae_fg",
])
csv_writer.writeheader()


## 9. Optimizer + scheduler

AdamW with weight_decay=0.05, β=(0.9, 0.999) per SimMIM Sec 4.1.1. Linear warmup + cosine decay. Grad clip 5.0 (matches the official SimMIM repo `config.py`; not in the paper).


In [ ]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay,
    betas=(0.9, 0.999),
)

def lr_lambda(epoch):
    if epoch < train_cfg.warmup_epochs:
        return (epoch + 1) / max(1, train_cfg.warmup_epochs)
    progress = (epoch - train_cfg.warmup_epochs) / max(1, train_cfg.epochs - train_cfg.warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

## 12. Training loop

In [ ]:
best_val_loss = float("inf")
best_epoch    = 0
total_start   = time.time()

for epoch in range(1, train_cfg.epochs + 1):
    # --- Train ---
    model.train()
    running = 0.0
    grad_norms = []
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f"E{epoch}/{train_cfg.epochs} train", leave=False)
    for batch_idx, x in enumerate(pbar, 1):
        x = x.to(device, non_blocking=True)
        x = augment_batch(x)
        mask = make_mask(x, train_cfg, TAU)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            recon = model(x, mask)
            loss  = reconstruction_loss(recon, x, mask, train_cfg.loss_kind, TAU, ALPHA)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_cfg.grad_clip_norm)
        grad_norms.append(float(gnorm))
        scaler.step(optimizer); scaler.update()

        running += loss.item()
        pbar.set_postfix(loss=f"{running/batch_idx:.5f}", g=f"{float(gnorm):.2f}")

    train_loss = running / len(train_loader)
    train_time = time.time() - t0
    scheduler.step()

    # --- Val ---
    model.eval()
    vrun = 0.0
    t1 = time.time()
    with torch.no_grad():
        for x in val_loader:
            x = x.to(device, non_blocking=True)
            mask = make_mask(x, train_cfg, TAU)
            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                recon = model(x, mask)
                loss  = reconstruction_loss(recon, x, mask, train_cfg.loss_kind, TAU, ALPHA)
            vrun += loss.item()
    val_loss = vrun / len(val_loader)
    val_time = time.time() - t1

    # --- Diagnostics every 5 epochs ---
    if epoch == 1 or epoch % 5 == 0 or epoch == train_cfg.epochs:
        feats = collect_diag_features(model, val_loader)
        d = collapse_metrics(feats)
        mae = recon_mae_split(model, val_loader, train_cfg, TAU)
    else:
        d = dict(eff_rank=float("nan"), r_max=0, n_dead=-1, D=-1,
                 std_min=float("nan"), std_med=float("nan"), std_max=float("nan"))
        mae = dict(mae_all=float("nan"), mae_fg=float("nan"), n_fg=-1)

    cur_lr   = scheduler.get_last_lr()[0]
    mean_g   = float(np.mean(grad_norms)) if grad_norms else float("nan")
    max_g    = float(np.max(grad_norms))  if grad_norms else float("nan")
    epoch_t  = train_time + val_time
    star = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss; best_epoch = epoch; star = " *"
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "val_loss": val_loss, "train_loss": train_loss,
            "model_cfg": vars(model_cfg), "train_cfg": vars(train_cfg),
            "TAU": TAU, "ALPHA": ALPHA,
        }, save_dir / "best_model.pt")

    print(
        f"E{epoch:3d} | tr {train_loss:.5f} | val {val_loss:.5f} | lr {cur_lr:.2e} "
        f"| g {mean_g:.2f} (max {max_g:.2f}) | mae_all {mae['mae_all']:.4f} mae_fg {mae['mae_fg']:.4f} "
        f"| eff_rank {d['eff_rank']:.1f}/{d['r_max']} | t {epoch_t:.1f}s{star}"
    )
    csv_writer.writerow(dict(
        epoch=epoch, train_loss=train_loss, val_loss=val_loss, lr=cur_lr,
        grad_norm_mean=mean_g, grad_norm_max=max_g, epoch_time_s=epoch_t,
        eff_rank=d["eff_rank"], r_max=d["r_max"], n_dead=d["n_dead"], std_med=d["std_med"],
        mae_all=mae["mae_all"], mae_fg=mae["mae_fg"],
    ))
    csv_file.flush()

csv_file.close()
print(f"\ndone in {time.time()-total_start:.1f}s | best val_loss {best_val_loss:.5f} at epoch {best_epoch}")


## 13. Post-training: load best, export encoder

In [ ]:
ckpt = torch.load(save_dir / "best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"loaded best checkpoint from epoch {ckpt['epoch']} (val={ckpt['val_loss']:.5f})")

# Export the SwinViT encoder for downstream finetune
encoder_path = save_dir / "encoder_swinvit.pt"
torch.save({
    "swinvit_state_dict": model.swinViT.state_dict(),
    "model_cfg": vars(model_cfg),
    "train_cfg": vars(train_cfg),
    "best_epoch": ckpt["epoch"],
    "best_val_loss": ckpt["val_loss"],
}, encoder_path)
print(f"encoder weights saved to {encoder_path}")


## 14. Final diagnostics + reconstruction visualisation

In [ ]:
final = collapse_metrics(collect_diag_features(model, val_loader))
final_mae = recon_mae_split(model, val_loader, train_cfg, TAU, n_batches=8)
print(f"final | eff_rank {final['eff_rank']:.1f}/{final['r_max']} "
      f"({final['eff_rank']/final['r_max']*100:.1f}% of max) "
      f"| dead {final['n_dead']}/{final['D']} "
      f"| mae_all {final_mae['mae_all']:.5f} | mae_fg {final_mae['mae_fg']:.5f}")

# Reconstructions
model.eval()
xb = next(iter(val_loader)).to(device)
mb = make_mask(xb, train_cfg, TAU)
with torch.no_grad(), torch.amp.autocast(device.type, enabled=device.type == "cuda"):
    rb = model(xb, mb).float()

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i in range(4):
    in_img    = xb[i].amax(0).cpu().numpy()
    masked_in = (xb[i] * (1 - mb[i])).amax(0).cpu().numpy()
    recon_img = rb[i].amax(0).cpu().numpy()
    err_img   = (rb[i] - xb[i]).abs().amax(0).cpu().numpy()
    for ax, im, ttl in zip(
        axes[i],
        [in_img, masked_in, recon_img, err_img],
        ["input", "masked", "recon", "|err|"],
    ):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)
        ax.set_title(ttl); ax.axis("off")
fig.tight_layout(); plt.show()


In [ ]:
# === V4-FINAL-RECON-VIZ ===
# Reuse `viz_recon()` from the helpers cell on the SAME 4 overfit + 3
# unseen patches that the overfit-recon-viz cell visualised. Comparing
# the two figures (this one vs the OVERFIT one above) shows what the
# full training run learned on top of -- or destroyed compared to --
# the 4-patch overfit baseline.
#
# A genuinely useful pretraining run should reconstruct the UNSEEN
# patches markedly better here than after the 4-patch overfit; the
# OVERFIT patches may look slightly worse here (the full run does not
# memorise them) but the in-mask MAE should still be sensible.
viz_recon(
    model, x_fixed, mask_fixed, overfit_cfg.patch_indices,
    suptitle=f"FINAL recon -- {x_fixed.shape[0]} TRAIN patches "
             f"(same indices as overfit-recon-viz)",
    header="\nFINAL on overfit TRAIN patches (same indices as overfit-recon-viz):",
)

viz_recon(
    model, x_other, mask_other, viz_other_indices,
    suptitle="FINAL recon -- 3 UNSEEN random train patches "
             "(same indices as overfit-recon-viz)",
    header="\nFINAL on UNSEEN random patches (same indices as overfit-recon-viz):",
)

## V3 Sanity checks (post-training)

Reload the best checkpoint and run inference on the same fixed indices used
in the pre-training checks. Compares input vs reconstruction vs per-pixel
error, and plots the training/val loss curves from `history.csv`.

In [ ]:
# === V3-SANITY-CHECKS POST ===
# Post-training sanity checks. Reloads the best checkpoint and visualises
# reconstruction quality on the same fixed indices used in the pre-train
# section, then on `unseen_idx` and `unseen_indices` (matches reference
# cells 38-40 of pretrain_simMIM_swin_v2_fixed.ipynb).
import matplotlib.pyplot as plt
import numpy as np
import torch

from types import SimpleNamespace
overfit_cfg    = SimpleNamespace(patch_indices=[3, 4, 50, 1000])
unseen_idx     = 100
unseen_indices = [200, 500, 1500, 2000]

# --- Reload best checkpoint
_best_path = save_dir / "best_model.pt"
if not _best_path.exists():
    # fall back to the explicit `best_ckpt` variable some training cells set
    try:
        _best_path = best_ckpt
    except NameError as e:
        raise FileNotFoundError(f"no best checkpoint found at {save_dir}") from e

ck = torch.load(_best_path, map_location=device, weights_only=False)
for _k in ("model_state_dict", "model",):
    if _k in ck:
        model.load_state_dict(ck[_k]); break
else:
    raise KeyError(f"no model state in checkpoint, keys={list(ck.keys())}")
_vl = ck.get("val_loss", ck.get("best_val_loss", float("nan")))
print(f"reloaded {_best_path.name}: epoch={ck.get('epoch','?')}  "
      f"val={float(_vl):.5f}")


def _resolve_fixed(ds, indices):
    base = getattr(ds, "_base", ds)
    return torch.stack([base[i] for i in indices]).to(device)


def _forward(x_target, seed_off):
    torch.manual_seed(cfg.seed + seed_off)
    model.eval()
    with torch.no_grad():
        mask_ = make_mask(x_target, train_cfg, TAU)
        out = model(x_target, mask_)
        recon_ = out[0] if isinstance(out, tuple) else out
    return recon_.float(), mask_


def _grid(x_target, recon, mask, indices, suptitle):
    """4-row grid: input | mask | recon | per-pixel |err| with cyan mask
    contour. Mirrors cells 39-40 of the reference notebook."""
    err = (recon - x_target).abs()
    n = x_target.shape[0]
    rows = 4 if mask is not None else 3
    fig, axes = plt.subplots(rows, n, figsize=(3.2 * n, 3.0 * rows))
    if n == 1:
        axes = axes[:, None]
    for j in range(n):
        o = x_target[j].clamp(0, 1).cpu()
        r = recon[j].clamp(0, 1).cpu()
        e = err[j].mean(0).cpu()
        rgb_o = o.permute(1, 2, 0).numpy() if o.shape[0] >= 3 else o[0].numpy()
        rgb_r = r.permute(1, 2, 0).numpy() if r.shape[0] >= 3 else r[0].numpy()
        axes[0, j].imshow(rgb_o, cmap="gray")
        axes[0, j].set_title(f"input idx={indices[j]}"); axes[0, j].axis("off")
        row_off = 0
        if mask is not None:
            m = mask[j, 0].cpu()
            axes[1, j].imshow(m.numpy(), cmap="gray", vmin=0, vmax=1)
            axes[1, j].set_title(f"mask cov={mask[j].mean().item():.2f}")
            axes[1, j].axis("off")
            row_off = 1
        axes[1 + row_off, j].imshow(rgb_r, cmap="gray")
        axes[1 + row_off, j].set_title("recon"); axes[1 + row_off, j].axis("off")
        hm = axes[2 + row_off, j].imshow(e.numpy(), cmap="hot")
        if mask is not None:
            # cyan contour at the mask boundary (matches reference cell 39)
            axes[2 + row_off, j].contour(
                mask[j, 0].cpu().numpy(), levels=[0.5],
                colors=["cyan"], linewidths=1.2,
            )
            in_m  = mask[j, 0].cpu()
            in_mae = (e * in_m).sum() / max(in_m.sum().item(), 1.0)
            out_mae = (e * (1.0 - in_m)).sum() / max((1.0 - in_m).sum().item(), 1.0)
            axes[2 + row_off, j].set_title(
                f"|err| in={in_mae.item():.3f}  out={out_mae.item():.3f}"
            )
        else:
            axes[2 + row_off, j].set_title(f"|err| mean={e.mean().item():.3f}")
        axes[2 + row_off, j].axis("off")
    plt.suptitle(suptitle)
    plt.tight_layout(); plt.show()


# --- (1) overfit indices on the val split (reference cell 38)
x_target = _resolve_fixed(ds_val, overfit_cfg.patch_indices)
recon, mask = _forward(x_target, seed_off=1)
err = (recon - x_target).abs()
mae_per = err.mean(dim=(1, 2, 3))
for j, idx in enumerate(overfit_cfg.patch_indices):
    print(f"  overfit_idx={idx}: MAE={mae_per[j].item():.5f}")
_grid(x_target, recon, mask, overfit_cfg.patch_indices,
      "POST: overfit indices on val split")


# --- (2) single unseen index (reference cell 38)
x_target = _resolve_fixed(ds_val, [unseen_idx])
recon, mask = _forward(x_target, seed_off=2)
print(f"  unseen_idx={unseen_idx}: MAE={(recon - x_target).abs().mean().item():.5f}")
_grid(x_target, recon, mask, [unseen_idx], f"POST: unseen idx={unseen_idx}")


# --- (3) unseen grid (reference cell 40)
x_target = _resolve_fixed(ds_val, unseen_indices)
recon, mask = _forward(x_target, seed_off=3)
err = (recon - x_target).abs()
for j, idx in enumerate(unseen_indices):
    print(f"  unseen_idx={idx}: MAE={err[j].mean().item():.5f}")
_grid(x_target, recon, mask, unseen_indices, "POST: unseen grid")


# --- (4) Loss / eff_rank curves from history.csv
import csv as _csv
hist_path = save_dir / "history.csv"
if hist_path.exists():
    rows = list(_csv.DictReader(open(hist_path)))
    if rows:
        ep = [int(r["epoch"]) for r in rows]
        tr = [float(r["train_loss"]) for r in rows]
        vl = [float(r["val_loss"])   for r in rows]
        fig, ax = plt.subplots(1, 2, figsize=(13, 4))
        ax[0].plot(ep, tr, label="train"); ax[0].plot(ep, vl, label="val")
        ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend()
        ax[0].set_title("Loss curves")
        if "eff_rank" in rows[0] and rows[0]["eff_rank"] not in ("", "nan"):
            er = [float(r["eff_rank"]) if r["eff_rank"] not in ("", "nan") else float("nan")
                  for r in rows]
            ax[1].plot(ep, er); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("eff_rank")
            ax[1].set_title("Effective rank (representation diversity)")
        plt.tight_layout(); plt.show()
else:
    print(f"history.csv not found at {hist_path}")


In [ ]:
# dataloader cleanup
import gc
for _name in ("train_loader", "val_loader"):
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
